[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-2_%EC%9D%B4%EC%83%81%ED%83%90%EC%A7%80_%EC%A0%95%EC%83%81%EC%9D%84%EB%B0%B0%EC%9A%B0%EA%B3%A0_%EB%B2%97%EC%96%B4%EB%82%A8%EC%B0%BE%EA%B8%B0.ipynb)

# 2-2 · 이상탐지 — 정상을 배우고 벗어남 찾기 (펌프 순환 시험장치 SKAB)

**⏱ 100분** · 모듈 2: 센서 → 특징 → **① 이상탐지** → ② 고장진단 → 판정 기준

## 🎯 이번 시간의 질문

> **정상 운전 기록만** 가지고, 처음 보는 이상(밸브 막힘, 회전자 불균형, 캐비테이션 …)에 경보를 울릴 수 있을까요? 오경보는 얼마나 날까요?

1-2에서는 "보류 섹터 > 0"이라는 기준을 바로 쓸 수 있었습니다. 정상 디스크는 0이라는 것을 알았기 때문입니다. 대부분의 설비는 그렇지 않습니다. 그래서 **정상일 때의 기록에서 "평소 모습"을 배우고, 거기서 벗어나면 경보**합니다. 이것이 **이상탐지(Anomaly Detection)**입니다. 고장 기록이 없어도 쓸 수 있다는 것이 가장 큰 장점입니다.

```
① 정상일 때의 기록으로 "평소 모습"을 배운다 → ② 새 기록이 평소에서 얼마나 먼지 점수를 매긴다 → ③ 점수가 기준을 넘으면 경보
```

이번 시간에는 2-1의 유압 장치가 아니라 **다른 장치**의 데이터를 씁니다. 왜 그런지부터 데이터로 확인합니다.

**이번 시간에 익힐 패턴:** P1 읽고 보기 · P3 `groupby`로 비교 · 평균·표준편차로 z-점수 · 교차표로 평가 · 다변량 거리(T²) `fit` → 점수

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/skab_normal.csv",
             "data/skab_experiments.csv.gz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 1. 왜 유압 장치 데이터로 하지 않나

이상탐지는 **건강한 장치**의 정상 기록으로 배워야 합니다. 2-1의 유압 장치에서 네 부품이 **모두** 정상인 사이클은 몇 개일까요? P2의 조건 네 개를 `&`로 묶어 셉니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
labels = pd.read_csv('data/hydraulic_labels.csv')
healthy = (labels['cooler'] == 100) & (labels['valve'] == 100) & (labels['pump_leakage'] == 0) & (labels['accumulator'] == 130)
print('건강한 장치의 사이클:', healthy.sum(), '/', len(labels))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 21 / 2205.

💬 **결과 해설** — 2,205사이클 중 **21개(1%)**뿐이고, 그중 11개는 상태를 바꾼 직후의 과도 구간입니다. 유압 장치 실험은 고장 조합 144가지를 골고루 시험하도록 설계되어서, 고장 상태의 정답은 풍부하지만 **건강한 장치의 기록은 거의 없습니다.** 이 정도로는 "평소 모습"을 배울 수도, 오경보가 얼마나 날지 확인할 수도 없습니다.
그래서 유압 장치 데이터는 정답을 이용하는 **고장진단**(2-3)에 쓰고, 이상탐지는 **정상 운전 기록이 충분한 장치**로 배웁니다. 데이터마다 잘 맞는 문제가 따로 있다는 것, 그리고 **PHM에서는 목적에 맞는 데이터를 얻는 것부터가 어렵다**는 것을 기억해 두세요.

## 2. 장치와 실험 (읽기, 5분)

러시아 Skoltech 연구소의 **물 순환 시험장치**(SKAB, Skoltech Anomaly Benchmark)입니다. 이상탐지 방법을 비교하려고 만든 공개 데이터입니다.

![SKAB 시험장치 (출처: Katser & Kozitsin, SKAB, GPL-3.0)](https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/data/skab_testbed.jpg)

```
물 탱크 ──▶ 입구 밸브 ──▶ 펌프 (전동기가 돌림) ──▶ 출구 밸브 ──▶ 배관 ──▶ 물 탱크   (물이 계속 순환)
                          │
            진동 센서 2개 · 전동기 전류·전압·몸체 온도 · 펌프 뒤 압력 · 물 온도 · 유량   ← 1초마다 기록
```

연구자들은 이렇게 기록했습니다.

1. **정상 운전만** 약 2.6시간(9,405초) 기록했습니다(2020년 2월 8일).
2. **이상 실험 34회**를 했습니다. 실험마다 약 20분 동안, **처음 약 9.5분은 정상으로** 돌리다가 연구자가 이상을 일으킵니다. 이상은 몇 분 이어진 뒤 끝납니다.
3. 1초마다 센서 8개를 기록하고, 그 1초가 **이상 중이었는지(anomaly = 1)** 정답을 붙였습니다. 실험은 2월 8일, 3월 1일, 3월 9일 사흘에 걸쳐 했습니다.

| 이상 종류 | 실험 수 | 무엇을 했나 |
|---|---|---|
| 입구 밸브 닫힘 | 16 | 펌프로 들어가는 쪽 밸브를 닫음 |
| 출구 밸브 닫힘 | 4 | 펌프에서 나가는 쪽 밸브를 닫음 |
| 누설·유입 | 4 | 물이 새거나 더해지는 상황을 흉내 냄 |
| 회전자 불균형 | 5 | 펌프 회전축에 불균형을 줌 (갑자기·서서히·계단식 등) |
| 물 증가 (서서히 / 갑자기) | 2 | 순환하는 물의 양을 늘림 |
| 캐비테이션 (물 빼기 / 기포 유입) | 2 | 펌프 입구에 물이 모자라거나 기포가 섞여 들어감 |
| 뜨거운 물 공급 | 1 | 더 뜨거운 물을 넣음 |

## 3. 데이터 구조 한눈에 보기 (읽기, 2분)

```
이 수업의 파일                               한 행                       크기
──────────────────────────────────           ─────────────────────       ──────────────
① skab_normal.csv       정상 운전 기록      1초                         9,405행 × 10열
② skab_experiments.csv  이상 실험 34회      실험 1회의 1초             37,401행 × 15열
                                             (exp_id로 실험을 구분, anomaly = 정답)
```

두 파일 모두 센서 8개가 같은 이름으로 들어 있습니다. 차례로 열어 봅니다.

## 4. ① 정상 운전 기록

**P1 · 읽고 보기**

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
normal = pd.read_csv('data/skab_normal.csv')
print(normal.shape)
normal.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 9,405행 × 10열, **한 행 = 1초**입니다.

| 열 | 뜻 | 단위 |
|---|---|---|
| `datetime`, `t_s` | 기록 시각, 기록 시작 후 초 | |
| `vib1`, `vib2` | 펌프 진동 (가속도 RMS) | g |
| `current`, `voltage` | 전동기 전류, 전압 | A, V |
| `pressure` | 펌프 뒤 배관 압력 | bar |
| `temp_motor`, `temp_water` | 전동기 몸체 온도, 순환하는 물 온도 | °C |
| `flow` | 순환 유량 | L/min |

## 5. ② 이상 실험 기록

**P1 · 읽고 보기**

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
exp = pd.read_csv('data/skab_experiments.csv.gz')
print(exp.shape)
exp.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 37,401행 × 15열, **한 행 = 실험 1회의 1초**입니다. 앞의 열이 실험 정보(`exp_id` 실험 번호, `file` 원본 파일, `kind` 이상 종류, `t_s` 실험 시작 후 초), 뒤의 열이 센서 8개와 정답(`anomaly` 이상 중이면 1, `changepoint` 이상이 시작·끝나는 순간이면 1)입니다.

**P3 · 묶어 요약** — 이상 종류별로 실험이 몇 번이고, 전체 시간 중 몇 %가 이상이었는지 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
exp.groupby('kind').agg(실험수=('exp_id', 'nunique'), 이상비율=('anomaly', 'mean')).round(2)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 이상 구간은 실험 시간의 약 30~50%입니다. 입구 밸브 닫힘 실험이 16회로 가장 많습니다.

실험 두 개를 그려 봅니다(▶ 실험 16: 입구 밸브 닫힘, 실험 27: 회전자 불균형). 빨간 영역이 정답이 이상인 구간입니다.

In [ ]:
# ▶ 그대로 실행하세요
def show_experiment(k, cols=('flow', 'vib1', 'current', 'temp_motor')):
    d = exp[exp['exp_id'] == k]
    fig, axes = plt.subplots(len(cols), 1, figsize=(11, 1.8 * len(cols)), sharex=True)
    for ax, c in zip(axes, cols):
        ax.plot(d['t_s'], d[c], lw=0.8)
        ax.fill_between(d['t_s'], d[c].min(), d[c].max(), where=d['anomaly'] == 1, color='red', alpha=0.15)
        ax.set_ylabel(c)
    axes[-1].set_xlabel('실험 시작 후 시간 (초)')
    fig.suptitle(f"실험 {k}: {d['kind'].iloc[0]} ({d['file'].iloc[0]})")
    fig.tight_layout()
    plt.show()

show_experiment(16)
show_experiment(27)

💬 **결과 해설** — 두 실험 모두 처음 약 570초는 정상입니다.
- **실험 16 (입구 밸브 닫힘)**: 밸브를 닫자 약 1분에 걸쳐 **유량**이 약 33 → 23 L/min으로 떨어집니다(이날은 유량이 낮아 센서 값이 1 L/min 단위로 찍힙니다). 진동과 전류에는 뚜렷한 변화가 없습니다. 전동기 온도는 처음 7분 동안 **서서히 오르다가**, 이상 표시(574초)보다 조금 앞선 480초 무렵부터 떨어집니다.
- **실험 27 (회전자 불균형)**: 반대로 **진동**이 계단처럼 크게 뛰고, 유량은 거의 그대로입니다.

**이상의 종류에 따라 반응하는 센서가 다릅니다.** 7절에서 숫자로 확인합니다.

## 6. 정상은 "언제의" 정상인가 — 운전 조건 비교

정상 기록(2월 8일)으로 배운 "평소 모습"을 모든 실험에 쓸 수 있을까요? 날짜별로 정상 구간의 센서 평균을 비교합니다. 실험은 **처음 400초**만 씁니다(모든 실험에서 이상이 일어나기 전).
`datetime`의 앞 10글자가 날짜입니다(`.str[:10]`).

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
normal['date'] = normal['datetime'].str[:10]
exp['date'] = exp['datetime'].str[:10]
print(normal.groupby('date')[['flow', 'vib1', 'current']].mean().round(2))
exp[exp['t_s'] < 400].groupby('date')[['flow', 'vib1', 'current']].mean().round(2)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 정상 기록(2월 8일) 유량 125.24 / 실험 처음 400초: 2월 8일 122.47 · 3월 1일 76.11 · 3월 9일 32.17.

💬 **결과 해설** — 같은 장치인데 날마다 유량이 약 125 → 76 → 32 L/min으로, 진동과 전류도 크게 다릅니다. 펌프 속도나 밸브 설정 같은 **운전 조건**이 날마다 달랐던 것입니다. 2월 8일의 "정상"으로 배우면 3월 9일은 정상 구간까지 모두 "평소와 다름"이 됩니다(9절에서 확인).
그래서 이번 시간에는 **실험마다 처음 400초(약 6.7분)를 그 실험의 정상**으로 삼아 배우고, 그 뒤를 채점합니다. 3-1에서 필터마다 처음 30초를 정상 기준으로 삼은 것과 같은 생각입니다.

**채점 방식** — 실험마다
- **오경보율** = 학습 구간(0~400초) 뒤, 이상이 시작되기 **전** 정상 구간에서 경보가 울린 초의 비율
- **탐지율** = 이상 구간에서 경보가 울린 초의 비율
- **발견** = 이상 구간에서 경보가 한 번이라도 울렸나, **발견까지** = 이상 시작 후 첫 경보까지 몇 초

(이상이 끝난 뒤의 구간은 장치가 곧바로 원래 상태로 돌아오지 않아서 채점에서 뺍니다. 과제 4에서 생각해 봅니다.)

---
## 7. 방법 1 · z-점수 — 센서 하나로 감시

실험 16에서 처음 400초의 **유량** 평균과 표준편차를 배우고, z-점수가 ±3을 넘으면 경보합니다(1-2의 "값 > 기준", 2-1의 구분력과 같은 "표준편차 몇 개만큼" 생각). 그리고 1-2와 같은 네 칸 교차표로 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
d = exp[exp['exp_id'] == 16]
train = d[d['t_s'] < 400]                          # 이 실험의 처음 400초 = 정상
mu, sigma = train['flow'].mean(), train['flow'].std()
z = (d['flow'] - mu) / sigma
alarm = z.abs() > 3
after = d['t_s'] >= 400                            # 채점은 학습 구간 뒤만
pd.crosstab(d['anomaly'][after], alarm[after], rownames=['실제 이상'], colnames=['경보'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 이상 401초 중 **307초(77%)**에 경보가 울리고, 정상 346초 중 헛경보는 **2초**뿐입니다(이 표는 이상이 끝난 뒤 구간까지 포함). 유량이 떨어지는 밸브 닫힘은 유량 하나로도 잘 잡힙니다. 다른 이상도 그럴까요?

### ▶ 34개 실험 전체를 채점하는 함수

1초 값은 흔들림이 커서, 함수에서는 **최근 5초 평균 점수**를 쓰고, 기준은 **학습 구간 점수의 99% 선**(정상도 100번 중 1번은 넘는 선)으로 정합니다. 방법(`method`)과 센서 목록(`cols`)만 바꿔 가며 같은 채점을 반복합니다.

In [ ]:
# ▶ 그대로 실행하세요
from sklearn.covariance import EmpiricalCovariance

def make_scorer(train, cols, method):
    """정상(train)으로 점수 함수를 만든다. 'z' = 센서 하나의 |z|, 'T2' = 센서 여러 개의 다변량 거리."""
    if method == 'z':
        mu, sd = train[cols[0]].mean(), train[cols[0]].std()
        return lambda df: ((df[cols[0]] - mu) / sd).abs().to_numpy()
    cov = EmpiricalCovariance().fit(train[cols])
    return lambda df: cov.mahalanobis(df[cols])

def check(cols, method='T2', q=0.99, n_train=400, detail=False):
    """34개 실험을 실험마다 처음 n_train초로 학습해 채점한다."""
    rows = []
    for k, d in exp.groupby('exp_id'):
        train = d[d['t_s'] < n_train]
        score = make_scorer(train, cols, method)
        smooth = lambda s: pd.Series(s).rolling(5, min_periods=1).mean().to_numpy()
        threshold = np.quantile(smooth(score(train)), q)
        alarm = smooth(score(d)) > threshold
        t, y = d['t_s'].to_numpy(), d['anomaly'].to_numpy()
        start = t[y == 1].min()
        found = t[alarm & (y == 1)]
        rows.append({'exp_id': k, 'kind': d['kind'].iloc[0],
                     '오경보율(%)': alarm[(t >= n_train) & (t < start)].mean() * 100,
                     '탐지율(%)': alarm[y == 1].mean() * 100,
                     '발견': len(found) > 0,
                     '발견까지(초)': found.min() - start if len(found) else np.nan})
    r = pd.DataFrame(rows)
    if detail:
        return r
    return pd.Series({'오경보율(%)': round(r['오경보율(%)'].mean(), 1), '탐지율(%)': round(r['탐지율(%)'].mean(), 1),
                      '발견': f"{r['발견'].sum()}/{len(r)}", '발견까지 중앙값(초)': r['발견까지(초)'].median()})

check(['flow'], 'z')

✅ **체크포인트** — 유량 z-점수: 오경보율 2.8%, 탐지율 67.6%, 발견 33/34, 발견까지 중앙값 24초.

이상 **종류별로** 나눠 봅니다. `detail=True`로 실험별 표를 받아 `groupby`로 묶습니다. 진동(`vib1`)과 나란히 놓습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
by_flow = check(['flow'], 'z', detail=True).groupby('kind')['탐지율(%)'].mean()
by_vib = check(['vib1'], 'z', detail=True).groupby('kind')['탐지율(%)'].mean()
pd.DataFrame({'유량 z': by_flow, '진동 z': by_vib}).round()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — **센서마다 잡는 이상이 다릅니다.** 유량은 밸브 닫힘(75~79%)과 물의 양 변화를 잘 잡지만 회전자 불균형은 35%입니다. 진동은 반대로 **회전자 불균형을 98%** 잡지만 밸브 닫힘은 5~9%입니다. 기포 유입(캐비테이션)은 둘 다 거의 못 잡습니다.
센서 하나만 보면 그 센서에 드러나는 이상만 잡힙니다. **센서 여러 개를 함께** 보면 어떨까요?

---
## 8. 방법 2 · T² — 센서 여러 개를 함께 (다변량 관리도)

센서 여러 개를 함께 보려면 "평균에서 표준편차 몇 개"를 하나로 정할 수 없습니다. **T²(Hotelling의 T², 마할라노비스 거리)**는 각 센서가 평균에서 벗어난 정도를 합치되, **센서끼리 함께 움직이는 관계까지** 고려한 거리입니다. 품질관리에서 배운 **다변량 관리도**와 같은 통계입니다.
예를 들어 정상일 때 유량과 전류가 함께 오르내린다면, 유량만 떨어지고 전류는 그대로인 순간은 두 값 각각은 범위 안이어도 **관계가 깨진** 것이라 T²가 커집니다.

7절과 같은 실험 16에서 처음 400초의 **센서 8개**로 정상의 평균과 관계(공분산)를 배우고(`fit`), 모든 시점의 T² 점수를 계산합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
S8 = ['vib1', 'vib2', 'current', 'pressure', 'temp_motor', 'temp_water', 'voltage', 'flow']
cov = EmpiricalCovariance().fit(train[S8])
t2 = cov.mahalanobis(d[S8])
plt.plot(d['t_s'], t2, lw=0.8)
plt.axhline(np.quantile(cov.mahalanobis(train[S8]), 0.99), color='red', linestyle='--', label='정상의 99% 선')
plt.fill_between(d['t_s'], 0, t2.max(), where=d['anomaly'] == 1, color='red', alpha=0.1, label='실제 이상')
plt.yscale('log')
plt.xlabel('실험 시작 후 시간 (초)')
plt.ylabel('T² 점수 (로그 눈금)')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 이상 구간에서 T²가 기준선보다 수십 배(중앙값 약 28배) 커집니다. 그런데 **이상이 시작되기 전(400~573초)에도 절반가량** 기준선을 넘습니다. 5절 그림에서 전동기 온도가 이상 표시보다 먼저 움직였기 때문입니다(정답 표시가 실제 변화와 수십 초 어긋날 수도 있다는 점도 기억해 두세요). 34개 실험 전체를 채점합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
check(S8, 'T2')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — T² 센서 8개: 오경보율 34.8%, 탐지율 90.2%, 발견 34/34, 발견까지 중앙값 1초.

💬 **결과 해설** — 34건을 모두, 거의 즉시 잡았지만 **정상 구간의 35%에 경보**가 울립니다. 쓸 수 없는 수준입니다. 5절의 그림에서 전동기 온도가 실험 내내 **서서히** 움직였던 것을 떠올려 보세요. 온도는 밸브 닫힘 같은 이상에도 반응하지만(실험 1은 이상 중 전동기 온도가 79 → 75°C), **정상일 때도 몇 분에 걸쳐 조금씩 변합니다.** 처음 400초 동안의 좁은 흔들림으로 "정상"을 배우니, 시간이 지나 조금만 달라져도 "평소와 다름"이 됩니다.

온도 2개를 빼고 **센서 6개**로 다시 채점합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
S6 = ['vib1', 'vib2', 'current', 'pressure', 'voltage', 'flow']
check(S6, 'T2')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — T² 센서 6개: 오경보율 6.3%, 탐지율 75.0%, 발견 34/34, 발견까지 중앙값 5초.

💬 **결과 해설** — 오경보가 35% → **6%**로 줄었고, 34건을 **모두** 발견했습니다(이상 시작 후 보통 5초 안). 대신 탐지율은 90% → 75%로 내려갔습니다. 온도가 가진 이상 신호도 함께 버렸기 때문입니다. 센서는 많이 넣을수록 좋은 것이 아니라, **정상일 때 어떻게 변하는지**를 보고 골라야 합니다(2-1 "데이터 의심하기", 3-1 "열화 신호 vs 운전 조건"과 같은 이야기). 온도를 살리려면 더 긴 기간의 정상으로 배우거나, 값 대신 **변화 속도**를 특징으로 쓰는 방법이 있습니다.

종류별로 봅니다(▶).

In [ ]:
# ▶ 그대로 실행하세요
r6 = check(S6, 'T2', detail=True)
r6.groupby('kind').agg(실험수=('exp_id', 'size'), 오경보율=('오경보율(%)', 'mean'), 탐지율=('탐지율(%)', 'mean'),
                       발견=('발견', 'sum'), 발견까지_중앙값=('발견까지(초)', 'median')).round(1)

💬 **결과 해설** — 이상 종류마다 다릅니다. 뜨거운 물·물의 양 변화·물 빼기(캐비테이션)·회전자 불균형은 86~99%의 시간에 경보가 울립니다. 밸브 닫힘은 71~74%이고, 발견까지 보통 25초쯤 걸립니다. 누설·유입(69%)과 회전자 불균형은 이상 전 정상 구간의 오경보가 12~15%로 높습니다. **기포 유입(캐비테이션)은 발견은 했지만 이상 시간의 3%에만 경보**가 울립니다. 기포 유입은 센서 6개의 평균 수준을 거의 바꾸지 않아서(이상 중 평균 변화가 모두 정상 표준편차의 1배 미만) 이 방법으로는 잡기 어렵습니다.

---
## 9. 정상을 잘못 정하면 — 다른 날의 정상으로 배울 때 (▶)

6절에서 날마다 운전 조건이 달랐습니다. 2월 8일 **정상 전용 기록(2.6시간)**으로 T²(센서 6개)를 배워 모든 실험에 적용하면 어떻게 될까요?

In [ ]:
# ▶ 그대로 실행하세요
cov_all = EmpiricalCovariance().fit(normal[S6])
smooth = lambda s: pd.Series(s).rolling(5, min_periods=1).mean().to_numpy()
th_all = np.quantile(smooth(cov_all.mahalanobis(normal[S6])), 0.99)
rows = []
for k, d_k in exp.groupby('exp_id'):
    alarm_k = smooth(cov_all.mahalanobis(d_k[S6])) > th_all
    y, t = d_k['anomaly'].to_numpy(), d_k['t_s'].to_numpy()
    rows.append({'날짜': d_k['date'].iloc[0], '오경보율(%)': alarm_k[t < t[y == 1].min()].mean() * 100, '탐지율(%)': alarm_k[y == 1].mean() * 100})
pd.DataFrame(rows).groupby('날짜').mean().round(1)

💬 **결과 해설** — 같은 날(2월 8일) 실험에서도 이상 전 정상 구간의 **57%**에 경보가 울리고, 다른 날(3월 1일·9일)은 **정상 구간의 100%**가 경보입니다. 모두 "평소와 다름"이라서 탐지율 100%는 아무 의미가 없습니다.
2.6시간이나 되는 정상 기록도 **다른 운전 조건**에는 쓸 수 없습니다. 이상탐지의 성능은 방법보다 **"정상"을 어떤 조건에서, 얼마나 모았느냐**가 먼저 정합니다.

---
## 10. 실습 과제 (35분)

### 📝 과제 1 · 기준을 바꾸면? (기준값 실험)
`check(S6, 'T2', q=...)`의 기준 분위수 q를 **0.95, 0.99, 0.999**로 바꿔 표를 채우세요. **실행 전에** 오경보율과 탐지율이 어떻게 변할지 예상해 보세요.

| q | 오경보율(%) | 탐지율(%) | 발견 | 발견까지 중앙값(초) |
|---|---|---|---|---|
| 0.95 | | | | |
| 0.99 | 6.3 | 75.0 | 34/34 | 5 |
| 0.999 | | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for q in [0.95, 0.999]: print(q, check(S6, 'T2', q=q).to_dict())


✏️ **나의 답:** 기준을 올리면 무엇을 얻고 무엇을 잃나요? 이 장치라면 어느 기준을 고르겠습니까?

### 📝 과제 2 · Isolation Forest와 비교하기
머신러닝 이상탐지 **Isolation Forest**는 데이터 공간에 무작위로 칸막이를 쳐서, **몇 번 만에 혼자 남는 점일수록 이상**으로 봅니다(분포를 가정하지 않음). 아래 함수로 센서 6개·8개에서 T²와 비교하세요.

| 방법 | 센서 6개: 오경보율 / 탐지율 / 발견 | 센서 8개(온도 포함): 오경보율 / 탐지율 / 발견 |
|---|---|---|
| T² | 6.3 / 75.0 / 34 | 34.8 / 90.2 / 34 |
| Isolation Forest | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: check_if(S6), check_if(S8)  — check()와 같은 채점, 점수만 Isolation Forest


✏️ **나의 답:** 어느 방법이 더 나은가요? 온도를 넣었을 때 두 방법은 어떻게 다르게 반응하나요?

### 📝 과제 3 · 여러 운전 조건의 정상을 한 모델로 (도전)
현장에서는 운전 조건마다 모델을 따로 두기 번거롭습니다. 34개 실험의 **처음 400초를 모두 모아** 모델 **하나**를 만들면 어떨까요? 세 날짜의 정상(유량 125 · 76 · 32)이 섞인 "여러 덩어리 정상"이 됩니다.
T²와, **가까운 이웃 몇 개와의 거리**로 판단하는 **LOF(Local Outlier Factor)**를 비교하세요. 채점은 이상 전 정상 구간(400초 이후)과 이상 구간입니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 아래 코드를 그대로 실행합니다. 모델 하나로 34개 실험을 모두 채점합니다.


✏️ **나의 답:** 정상이 "여러 덩어리"일 때 T²와 LOF는 어떻게 다른가요? 왜 그럴까요?

### 📝 과제 4 · 생각해 보기 — 현장에서 정상 데이터를 어떻게 모을까
1) 오늘 결과를 근거로, 펌프 설비에 이상탐지를 도입할 때 **정상 데이터를 모으는 계획**을 세 가지 적으세요(언제, 어떤 조건에서, 얼마나).
2) 이상이 끝난 뒤의 구간을 채점에서 뺐습니다. 실험 1의 그림(5절)을 보고, 이상이 끝난 뒤 장치가 곧바로 원래 상태로 돌아오지 않는다면 현장에서 어떤 문제가 생길지 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 이상탐지의 원리 | 정상 기록으로 "평소 모습"을 배움 → 점수 → 기준을 넘으면 경보. **고장 기록이 없어도** 된다 |
| 데이터 선택 | 유압 장치는 건강한 기록이 21개(1%)뿐 → 이상탐지는 정상 기록이 충분한 장치(SKAB)로, 유압 장치는 진단(2-3)으로 |
| 정상의 정의 | 날마다 운전 조건이 다름(유량 125 · 76 · 32) → 같은 조건의 정상으로 배워야 한다. 다른 날의 정상으로 배우면 정상 구간도 100% 경보 |
| z-점수 (센서 하나) | 센서마다 잡는 이상이 다르다 (유량 → 밸브, 진동 → 회전자 불균형) |
| T² (다변량 관리도) | 센서끼리의 관계까지 본다. 천천히 변하는 온도를 빼면 오경보 6% · 34건 모두 발견(보통 5초 안) |
| 평가 | 1-2와 같은 네 칸 → 오경보율·탐지율, 그리고 발견 여부·발견까지 시간 |
| 교환 관계 | 기준을 올리면 오경보↓ 탐지↓ (과제 1) |

다음 질문: 이상탐지는 "평소와 다르다"는 것만 알려 줍니다. **어느 부품이 얼마나** 나빠졌는지 알려면? 정답 기록이 풍부한 유압 장치로 돌아갑니다 → **2-3 · 고장진단**